
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 랩 - 문서 검색을 위한 AI Search 구축

랩에 오신 것을 환영합니다! 아래 단계를 따라 Databricks을 사용해 문서 검색용 AI Search 솔루션을 만드는 방법을 배워보세요.

## 개요

이 실험실에서는 Parquet 파일에 저장된 문서 조각들을 다루며 완전한 AI Search 솔루션을 구축할 것입니다. 데이터를 **준비하는**, **AI Search 인덱스 생성**, 그리고 Databricks AI Search 기능을 사용해 **다양한 유형의 검색**을 수행하는 방법을 배우게 됩니다.

## 학습 목표

이 실험실이 끝날 때쯤이면 다음과 같은 일을 할 수 있게 될 것입니다:
1. Parquet 데이터를 **읽고** change data feed 활성화된 상태에서 Delta 테이블로 저장하세요.
1. Databricks UI를 사용해 AI Search 인덱스를 **생성**하세요.
1. 검색 수행을 위한 AI Search 인덱스를 **받으십시오**.
1. 정확도를 높이기 위해 유사성 검색과 리랭킹을 함께 **구현**하세요.
1. 필터링이 포함된 하이브리드 **검색**을 실행해 특정 문서를 타겟팅하세요.

## 요구 사항
- 미리 생성된 **AI Search 엔드포인트**. 이것은 미리 생성되었습니다.
- **서버리스 Compute (환경 버전 5)**. [여기](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version)에 따라 적절한 환경 버전을 선택하세요.
- 서버리스 compute 구성의 **의존성**에 필요한 라이브러리가 추가됩니다.
- AI Search 인덱스를 생성하고 관리할 수 있는 적절한 권한 부여.
- 임베딩 생성을 위한 Foundation Model APIs 접근.


**📌 당신의 태스크: 이 실험실에서 당신의 태스크는 섹션을 적절한 코드로 교체 `<FILL_IN>` 하는 것입니다.**

## 준비

아래 코드를 실행하여 필요한 라이브러리를 설치하고 교실 환경을 구성하세요. 이 단계는 모든 의존성이 사용 가능하고 워크스페이스가 데모 준비가 완료되도록 보장합니다.

In [0]:
%run ../Includes/Classroom-Setup-03

## 태스크 1: Parquet 데이터를 읽고 CDF로 Delta 표를 만드세요

이 섹션에서는 Parquet 파일에서 문서 청크를 읽어 change data feed(CDF)를 활성화한 Delta 테이블로 저장합니다. CDF는 AI Search 동기화를 위해 필요합니다.

**단계:**
1. 문서 조각이 포함된 Parquet 파일을 Pandas로 읽으세요.
2. Spark DataFrame으로 변환한 후 Delta 테이블로 저장하세요.
3. 테이블에서 Change Data Feed을 활성화하세요.
4. 표 구조를 검증하기 위해 샘플 데이터를 표시하세요.

아래 코드를 작성하여 이 작업을 수행하세요.

In [0]:
docs_chunked_lab_3 = f"{catalog}.{schema}.docs_chunked_lab_3"

In [0]:
## Parquet 파일을 읽고 CDF가 활성화된 Delta 테이블을 생성합니다
import os
import pandas as pd

## Parquet 파일 경로를 정의하세요
parquet_path = f"/Volumes/{catalog}/{schema}/orion_text/docs_chunked.parquet"

## Parquet 파일을 Pandas로 읽습니다
pdf = <FILL_IN>

## it이 이미 존재한다면 테이블을 삭제해 충돌을 피하세요
spark.sql(f"DROP TABLE IF EXISTS {docs_chunked_lab_3}")

## Pandas DataFrame를 Spark DataFrame로 변환하고 Unity Catalog로 쓰세요
df = spark.createDataFrame(pdf)
df.write.<FILL_IN>

## AI Search 동기화를 위해 change data feed 활성화
spark.sql(f"<FILL_IN>")

In [0]:
%skip
# Parquet 파일을 읽고 CDF가 활성화된 Delta 테이블 생성
import os
import pandas as pd

# Parquet 파일 경로를 정의하세요
parquet_path = f"/Volumes/{catalog}/{schema}/orion_text/docs_chunked.parquet"

# Read Parquet file using pandas
pdf = pd.read_parquet(parquet_path)

# 이미 존재한다면 테이블을 내려서 갈등을 피하세요
spark.sql(f"DROP TABLE IF EXISTS {docs_chunked_lab_3}")

# Pandas DataFrame를 Spark DataFrame로 변환하고 Unity Catalog로 쓰세요
df = spark.createDataFrame(pdf)
df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(docs_chunked_lab_3)

# AI Search 동기화를 위해 change data feed 활성화
spark.sql(f"ALTER TABLE {docs_chunked_lab_3} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"👍 Table '{docs_chunked_lab_3}' created with Change Data Feed enabled.")

In [0]:
# 표 구조를 이해하기 위해 샘플 데이터를 표시하세요
display(spark.sql(f"SELECT * FROM {docs_chunked_lab_3} LIMIT 5"))

## 태스크 2: UI를 이용해 AI Search 인덱스를 생성하세요

이 섹션에서는 **Databricks UI를 사용해 AI Search 인덱스를 생성할 것입니다**. 이 방법은 관리되는 임베딩으로 인덱스를 구성할 수 있는 사용자 친화적인 인터페이스를 제공합니다.

**UI를 통한 인덱스 생성 단계:**

1. 왼쪽 사이드바에서 **카탈로그 (Catalog)** 를 클릭하여 Catalog Explorer를 엽니다.
2. 카탈로그와 스키마로 이동하세요.
3. Delta 테이블(`docs_chunked_lab_3`) 을 찾아 선택하십시오.
4. **생성 (Create)** (오른쪽 상단)를 클릭하고 **AI Search 인덱스**를 선택하세요.
5. 대화 상자에서 다음 설정을 설정하세요:
   * **이름:** 인덱스 이름을 `docs_chunked_lab_index`로 입력하십시오.
   * **인덱스 유형** `Hybrid`
   * **기본 키:** `id` (고유 식별자 열)을 선택하십시오
   * **임베딩 소스:** **임베딩 컴퓨트 (Compute embeddings)** 을 선택하세요
     - **Embedding source column:** `chunk` 선택하세요
     - **Embedding model:** `databricks-gte-large-en` 선택하세요
   * **계산된 임베딩:** 테이블에 임베딩을 저장하려면 **OFF**로 설정하세요
   * **AI Search 엔드포인트:** 엔드포인트를 선택하세요. 참고: 이 엔드포인트가 준비되어 있어야 합니다.
   * **동기화 모드:** **Triggered** (수동 동기화)를 선택하세요.
   * **고급 설정(Advanced settings):**
      - **색인할 열(Columns to index):** (표준 엔드포인트만 해당) 포함할 열을 선택하거나, 모든 열을 동기화하려면 비워 둡니다.
6. **생성 (Create)** 를 클릭하고 인덱스 생성 진행 상황을 모니터링하세요.

**⏱️ 대기 시간:** 인덱스 생성은 보통 2-3분 정도 소요됩니다. UI에서 진행 상황을 모니터링할 수 있습니다.

인덱스가 생성되면 다음 태스크로 넘어가세요.

## 태스크 3: AI Search 인덱스 세부 정보 보기

방금 만든 AI Search 인덱스를 받아 세부 정보를 보여주세요.

In [0]:
## 검색 수행을 위한 AI Search 인덱스를 얻으세요

from databricks.vector_search.client import VectorSearchClient

## UI를 통해 만든 인덱스 이름을 정의하세요
index_name = f"{catalog}.{schema}.docs_chunked_lab_index"

## 나중에 사용할 AI Search 클라이언트를 초기화하세요
vsc = VectorSearchClient(disable_notice=True)

## AI Search 클라이언트를 사용해 인덱스를 얻으세요
index = vsc.<FILL_IN>

## 인덱스 정보 표시
print(index.describe())

In [0]:
%skip
# 검색 수행을 위한 AI Search 인덱스를 얻으세요
from databricks.vector_search.client import VectorSearchClient

# UI를 통해 만든 인덱스 이름을 정의하세요
index_name = f"{catalog}.{schema}.docs_chunked_lab_index"

# AI Search 클라이언트를 초기화하여 나중에 사용하세요
vsc = VectorSearchClient(disable_notice=True)

# AI Search 클라이언트로 인덱스를 얻기
index = vsc.get_index(index_name=index_name)

# 색인 정보 표시
print(index.describe())

## 태스크 4: 리랭킹을 이용한 유사성 탐색

이 섹션에서는 검색 정밀도를 높이기 위해 **유사성 검색과 재순위를 수행**할 것입니다. 재순위 조정은 가장 맥락적으로 적합한 결과를 우선순위로 정하기 위해 2차 점수 단계로 적용됩니다.

**단계:**
1. 정밀도를 높이기 위해 유사성 검색과 재랭킹을 수행하세요.
1. 이 질문을 해보세요: `"How does the motion controller maintain balance during rapid movement?"`
1. 3개의 결과를 반환하세요.
1. 결과를 분석하여 재순위 조정의 영향을 이해하세요.

아래 코드를 작성하여 이 작업을 수행하세요.

In [0]:
## 정확도를 높이기 위해 리랭킹을 포함한 유사성 탐색을 수행하세요

Databricks에서.vector_search.reranker import DatabricksReranker

query_text = "How does the motion controller maintain balance during rapid movement?"

## 리랭킹과 유사성 탐색 수행
reranked_results = index.<FILL_IN>

print("=== Similarity Search with Reranking Results ===")
display(reranked_results)

In [0]:
%skip
## 정확도 향상을 위해 리랭킹을 통한 유사성 검색 수행

from databricks.vector_search.reranker import DatabricksReranker

query_text = "How does the motion controller maintain balance during rapid movement?"

## 재순위 적용 유사성 검색 수행
reranked_results = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    num_results=3,
    reranker=DatabricksReranker(columns_to_rerank=["chunk"])
)

print("=== Similarity Search with Reranking Results ===")
display(reranked_results)

## 태스크 5: 필터가 포함된 하이브리드 검색 – 타겟 문서 검색

이 섹션에서는 의미적 유사성, 키워드 매칭, 문서 타겟팅을 결합하는 **필터가 포함된 하이브리드 검색을 구현**합니다. 이 접근법은 여러 검색 전략을 함께 활용하여 매우 정밀한 결과를 제공합니다.

답변하려면: **"A1 모델의 배터리 교체 절차를 찾아보세요."**

- *순수 의미(유사성) 검색*은 배터리 충전이나 열 관리에 관한 구절을 반환할 수 있는데, 이는 의미적으로 관련되어 있기 때문입니다.
- *"배터리" 또는 "A1"에 대한 키워드 필터* 추가는 관련 문서 섹션으로 결과를 좁힙니다.
- *파일 이름으로 필터링*을 사용하면 올바른 문서에서 절차만 검색할 수 있습니다.

**단계:**
1. 쿼리에 대해 **하이브리드 검색**을 수행하세요.
1. 필터링하여 `05_Orion_Maintenance_and_Servicing_Guide_v3.pdf`에서만 결과를 표시합니다.
1. **2개의 레코드**와 **모든 열**을 반환하세요.
1. 하이브리드 검색과 필터를 결합하면 검색 정밀도가 어떻게 향상되는지 분석하세요.

아래 코드를 작성하여 이 작업을 수행하세요.

In [0]:
## 필터링이 포함된 하이브리드 검색을 수행하여 특정 문서를 대상으로 합니다.

query_text = "Find procedures that describe battery replacement for the A1 model."

## 문서 경로 필터를 이용한 하이브리드 검색 수행
filtered_hybrid_results = index.<FILL_IN>

print("=== Hybrid Search with Filters Results ===")
display(filtered_hybrid_results)

In [0]:
%skip
# 필터링이 포함된 하이브리드 검색을 수행하여 특정 문서를 대상으로 합니다

query_text = "Find procedures that describe battery replacement for the A1 model."

# 문서 경로 필터를 이용한 하이브리드 검색 수행
filtered_hybrid_results = index.similarity_search(
    query_text=query_text,
    columns=["id","path", "chunk"],
    query_type="hybrid",
    filters={"path LIKE": "05_Orion_Maintenance_and_Servicing_Guide_v3.pdf"},  # Filter by path containing safety-related documents
    num_results=5
)

print("=== Hybrid Search with Filters Results ===")
display(filtered_hybrid_results)

**💡 분석 및 성찰:**

**어떤 검색 방법을 선택하시겠습니까:**
- 사용자가 특정 절차를 검색하는 기술 문서 시스템?
- 정밀함이 중요한 법적 저장소?
- 다양한 쿼리 유형이 있는 고객 지원 지식 기반?

**생각해보세요:** 결과를 바탕으로 프로덕션 RAG 시스템에 어떤 접근법을 추천하시겠습니까? 그 이유는 무엇입니까?

## 요약과 다음 단계

Databricks를 이용한 문서 검색용 AI Search 솔루션을 만들기 위한 랩을 완료하셨습니다. 다음을 학습하셨습니다:

* 문서 데이터를 **준비**하려면 Parquet에서 읽고 change data feed가 활성화된 Delta 테이블을 생성합니다.
* AI Search 인덱스를 **생성**하려면 관리된 임베딩을 사용해 Databricks UI를 사용하세요.
* 유사성 검색을 **구현**하여 검색 정확도를 높이기 위해 리랭킹을 사용하세요.
* 하이브리드 검색을 **실행**하여 의미적 유사성과 키워드 매칭을 결합하세요.
* 필터를 **적용**하여 특정 문서를 대상으로 하고 검색 범위를 좁히세요.

**다음 단계 (선택 사항):**
* 다양한 임베딩 모델을 탐험하고 **다국어 및 도메인 특화** 임베딩 모델을 실험하세요.
* 엔드포인트 구성, 임베딩 차원, 갱신 모드가 지연 시간과 비용에 미치는 영향을 조사하십시오.
* 사용자의 자격 증명을 기반으로 행을 복구하는 방법을 조사하세요. 

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>